In [1]:
from transformers import pipeline


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/Users/ahmedmohamed/miniforge3/lib/python3.9/runpy.py", line 197, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/Users/ahmedmohamed/miniforge3/lib/python3.9/runpy.py", line 87, in _run_code
    exec(code, run_globals)
  File "/Users/ahmedmohamed/miniforge3/lib/python3.9/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/Users/ahmedmohamed/miniforge3/lib/python3.9/site-packages/traitlets/config/application.py", line 1043, in launch_instan

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/Users/ahmedmohamed/miniforge3/lib/python3.9/runpy.py", line 197, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/Users/ahmedmohamed/miniforge3/lib/python3.9/runpy.py", line 87, in _run_code
    exec(code, run_globals)
  File "/Users/ahmedmohamed/miniforge3/lib/python3.9/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/Users/ahmedmohamed/miniforge3/lib/python3.9/site-packages/traitlets/config/application.py", line 1043, in launch_instan

AttributeError: _ARRAY_API not found

ImportError: cannot import name 'pipeline' from 'transformers' (/Users/ahmedmohamed/miniforge3/lib/python3.9/site-packages/transformers/__init__.py)

In [1]:
import cv2 
frames = [] 

cap = cv2.VideoCapture('/Users/ahmedmohamed/Desktop/basketball/inputs/input_1.mp4') 
while True : 
    ret , frame = cap.read() 
        
    if not ret : 
        break 

    frames.append(frame) 
cap.release() 


In [29]:
import pickle
import os  
with open(os.path.join(os.getcwd() , 'presaved/player_ball.pkl') , 'rb') as f : 
    detections = pickle.load(f) 



In [30]:
detects  = detections[:238]

In [28]:
detects[0]

Detections(xyxy=array([[ 775.,  410.,  874.,  627.],
       [ 269.,  464.,  393.,  705.],
       [  92.,  212.,  170.,  395.],
       [1384.,  465., 1513.,  703.],
       [1110.,  686., 1191.,  902.],
       [1359.,  216., 1424.,  406.],
       [1337.,  629., 1437.,  863.],
       [ 501.,  516.,  592.,  746.],
       [ 671.,  172.,  789.,  333.],
       [1300.,  510., 1380.,  743.],
       [ 294.,  537.,  396.,  772.],
       [ 870.,  217.,  927.,  415.],
       [1231.,  386., 1341.,  557.],
       [ 333.,  523.,  363.,  553.],
       [1440.,  523., 1458.,  547.],
       [1269.,  183., 1329.,  207.],
       [ 395.,  535.,  465.,  745.],
       [ 527.,  568.,  550.,  600.],
       [1389.,  679., 1407.,  707.],
       [   0.,  653.,   38.,  898.]]), mask=None, confidence=array([0.93439519, 0.93318772, 0.93121266, 0.9296962 , 0.92603666,
       0.91993076, 0.91444367, 0.91180992, 0.90260971, 0.89588761,
       0.88523519, 0.8822251 , 0.8614217 , 0.86130512, 0.82615149,
       0.82090056, 

In [ ]:
import pandas as pd 
import numpy as np 
from utils import distance_between_two_points ,get_bottom_center_of_player
def tracker_hallosination(detections) : 
    max_detection_num = max([len(detection.xyxy) for detection in detections]) 
    player_detection_classes_ids =  [1,3,4,5,6,7]
    previous_tracker_ids = None 
    previous_player_boxes = None 
    for  i , detection in enumerate(detections): 
        # print(f'frame_{i}')
        filtered_ids = []
        filtered_ids_indexes = []
        filtered_player_boxes = []
        confidancesss = []
        for i , (tracker_id , class_id , box , conf) in enumerate(zip(detection.tracker_id , detection.class_id , detection.xyxy , detection.confidance) ): 
            if class_id in player_detection_classes_ids : 
                filtered_ids.append(tracker_id)
                filtered_player_boxes.append(box)
                filtered_ids_indexes.append(i)
                confidancesss.append(conf)
        if not any(track_id > max_detection_num for track_id in filtered_ids) :
            previous_tracker_ids = filtered_ids 
            previous_player_boxes = filtered_player_boxes 
        else : 
            print(f'frame_{i}')
            print(previous_tracker_ids)
            print(filtered_ids)
            mis_tracked = list(set(filtered_ids) - set(previous_tracker_ids)) 
            targets_for_comparasion = list(set(previous_tracker_ids) - set(filtered_ids)) 
            if not targets_for_comparasion : 
                for val in mis_tracked : 
                    filtered_player_boxes.pop(filtered_ids.index(val))
                    ind = np.where(detection.tracker_id == val)
                    detection.xyxy = np.delete( detection.xyxy,  ind , axis = 0) 
                    filtered_ids.remove(val) 
                    detection.tracker_id = detection.tracker_id[detection.tracker_id!=val]
                previous_tracker_ids = filtered_ids 
                previous_player_boxes = filtered_player_boxes
                continue

            mis_tracked_indexes = [filtered_ids.index(element) for element in mis_tracked] 
            targets_for_comparasion_indexes = [previous_tracker_ids.index(element) for element in targets_for_comparasion]
            distances = [] 
            for mis_tracked_index in mis_tracked_indexes : 
                mis_tracked_index_distances = []
                for targets_for_comparasion_index in targets_for_comparasion_indexes : 
                    mis_tracked_index_distances.append(distance_between_two_points( get_bottom_center_of_player(filtered_player_boxes[mis_tracked_index]) , get_bottom_center_of_player(previous_player_boxes[targets_for_comparasion_index])) )
                distances.append(mis_tracked_index_distances) 
            indexes_min_distances = [np.argmin(d) for d in distances] 
            correction_ids = [previous_tracker_ids[index] for index in indexes_min_distances]
            corrected_tracker_ids = filtered_ids
            corrected_player_boxes = filtered_player_boxes 
            for mis_tracked_index , correction_id in zip(mis_tracked_indexes , correction_ids) : 
                corrected_tracker_ids[mis_tracked_index] = correction_id
                corrected_player_boxes[mis_tracked_index] = previous_player_boxes[previous_tracker_ids.index(correction_id)]
            previous_tracker_ids = corrected_tracker_ids 
            previous_player_boxes = corrected_player_boxes
            for filtered_ids_index , corrected_tracker_id in zip(filtered_ids_indexes,previous_tracker_ids) : 
                detection.tracker_id[filtered_ids_index] =  corrected_tracker_id 
            
            # print(detection.xyxy) 
            # print(detection.tracker_id)
            print('-----------------')
        


In [27]:
tracker_hallosination(detects)

frame_18
[np.int64(2), np.int64(1), np.int64(8), np.int64(13), np.int64(7), np.int64(12), np.int64(9), np.int64(11), np.int64(4), np.int64(10)]
[np.int64(1), np.int64(2), np.int64(13), np.int64(7), np.int64(12), np.int64(8), np.int64(4), np.int64(11), np.int64(9), np.int64(86), np.int64(10)]
frame_18
[np.int64(1), np.int64(2), np.int64(13), np.int64(7), np.int64(12), np.int64(8), np.int64(4), np.int64(11), np.int64(9), np.int64(10)]
[np.int64(1), np.int64(2), np.int64(13), np.int64(11), np.int64(8), np.int64(12), np.int64(4), np.int64(9), np.int64(10), np.int64(86), np.int64(7)]
frame_19
[np.int64(7), np.int64(4), np.int64(10), np.int64(2), np.int64(13), np.int64(1), np.int64(9), np.int64(8), np.int64(11)]
[np.int64(2), np.int64(4), np.int64(7), np.int64(10), np.int64(13), np.int64(1), np.int64(9), np.int64(8), np.int64(11), np.int64(131)]
frame_21
[np.int64(2), np.int64(4), np.int64(7), np.int64(10), np.int64(13), np.int64(1), np.int64(9), np.int64(8), np.int64(11)]
[np.int64(2), np.i